In [ ]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages, MessagesState
from typing import Annotated
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config = {"configurable": {"thread_id": "1"}}

In [ ]:
class State(MessagesState):
    # MessagesState에 아래와 동일한 항목이 있음
    # messages: Annotated[list[AnyMessage], add_messages]
    custom_stuff: str

graph_builder = StateGraph(State)

@tool
def get_human_feedback(poem: str):
    """
    Asks the user for feedback on the poem.
    Use this before returning the final response.
    """
    # interrupt는 파이썬의 input과 유사하게 동작한다. 즉 해당 질문에 대한 대답을 user에게 받을 동안 그래프의 진행을 멈춰둔다
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback

llm_with_tools = llm.bind_tools(tools=[get_human_feedback])

def chatbot(state: State):
    response = llm.invoke(f"""
        You are an expert in making poems.
                          
        Use the `get_human_feedback` tool to get feedback on your poem.
        
        Only after you receive positive feedback you can return the final poem.
        
        ALWAYS ASK FOR FEEDBACK FIRST.
                          
        Here is the conversation history: {state["messages"]}
    """)
    return { "messages": [response] }

# tool을 노드에 등록
tool_node = ToolNode(
    tools=[get_human_feedback]
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
# tools_condition은 tools로 이동하거나 END로 이동하는 2가지 옵션이 있음
graph_builder.add_conditional_edges("chatbot", tools_condition)
# 이렇게 하는 이유는 chatbot에서 llm을 호출하고 그 결과에 따라 tools를 호출 -> 이후 tools가 다시 chatbot을 호출해야 하기 때문!
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    # 이렇게 해서 db 연결이 가능함
    checkpointer=SqliteSaver(conn)
)

In [ ]:
graph.invoke(
    {"messages": [ {"role": "user", "content": "What is the weather in korea"} ]},
    # recursion_limit는 chatbot과 tools의 상호작용을 몇번으로 제한할 것인지에 대한 설정 (무한루프 방지 가능)
    config=config
)

async for event in graph.astream(
    {"messages": [ {"role": "user", "content": "What did i just ask you about?"} ]},
    # stream 형식으로 받을 수 있는 방법
    stream_mode="messages"
    # config={"configurable": {"thread_id": "2"}}
):
    print(event)

for state in graph.get_state_history(
    {
        "configurable": {
            "thread_id": "2"
        }
    }
):
    print(state.next)

In [ ]:
result = graph.invoke({"messages": [{"role":"user", "content":"please make a poem about Python code."}]}, config=config)

for message in result["messages"]:
    message.pretty_print()

In [ ]:
# 그래프의 현재 상태를 확인하는 방법
snapshot = graph.get_state(config)

snapshot.next

In [ ]:
# command의 resume에 뭘 작성하든 get_human_feedback의 interrupt의 결과가 된다. 객체도 가능!
response = Command(
    resume="It looks good!"
)

# 이렇게 command의 결과를 넣을수도 있다
result = graph.invoke(response, config=config)

for message in result["messages"]:
    message.pretty_print()